# Adaptive Portfolio Positioning under Market Volatility

**Appendix: Daily Unrestricted Three-State Diagnosis**

This notebook documents an unrestricted daily three-state Markov-switching specification.  

The exercise shows that free multi-state models tend to produce unrealistically short (near unit) regime durations. This finding motivates the main paper’s choice of a two-state model combined with asymmetric soft persistence penalties.

**This is diagnostic material only — not part of the primary results.**


## 0. Notes

In [ ]:
# Adjust the following
# Date


## 1. Environment

In [ ]:
# Environment Setup, Imports, and Plotting Configuration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import cm
import seaborn as sns
from scipy.stats import norm, t as student_t
from scipy.optimize import minimize, differential_evolution
from scipy.special import logsumexp
import warnings
warnings.filterwarnings("ignore")

# Fixed random seed for full reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Global plotting style used throughout
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "figure.figsize": (14, 7),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Environment ready. Random seed fixed at", RANDOM_SEED)


## 2. Data

In [ ]:
# Robust Data Download & Alignment (No Look-Ahead Bias)

import yfinance as yf

# Download each series separately to avoid MultiIndex column problems
spy_data   = yf.download("SPY",   start="2010-01-04", end="2026-07-31",
                         auto_adjust=True, progress=False)["Close"]
vix_data   = yf.download("^VIX",  start="2010-01-04", end="2026-07-31",
                         auto_adjust=True, progress=False)["Close"]
vix3m_data = yf.download("^VIX3M", start="2010-01-04", end="2026-07-31",
                         auto_adjust=True, progress=False)["Close"]
bil_data   = yf.download("BIL",   start="2010-01-04", end="2026-07-31",
                         auto_adjust=True, progress=False)["Close"]

# Align all series on common trading days
price_data = pd.concat(
    [spy_data, vix_data, vix3m_data, bil_data],
    axis=1,
    join="inner"
)
price_data.columns = ["SPY", "VIX", "VIX3M", "BIL"]
price_data = price_data.dropna()

# Daily simple returns
price_data["SPY_return"] = price_data["SPY"].pct_change()
price_data["BIL_return"] = price_data["BIL"].pct_change()

# VIX term-structure inversion ratio, lagged one day (strictly no look-ahead)
price_data["Inversion"] = price_data["VIX"] / price_data["VIX3M"]
price_data["Inversion_lagged"] = price_data["Inversion"].shift(1)

# Drop first observation created by lag / differencing
price_data = price_data.dropna()

price_data.to_csv("data.csv")


## 3. Unrestricted Three-State Model

In [ ]:
# Exploratory Data Analysis and Descriptive Plots

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Panel A: SPY price
axes[0].plot(price_data.index, price_data["SPY"], color="navy", linewidth=1.2)
axes[0].set_ylabel("SPY Price")
axes[0].set_title("S&P 500 ETF (SPY) Adjusted Close Price")

# Panel B: VIX and VIX3M
axes[1].plot(price_data.index, price_data["VIX"], label="VIX", color="crimson", linewidth=1.0)
axes[1].plot(price_data.index, price_data["VIX3M"], label="VIX3M", color="darkorange", linewidth=1.0)
axes[1].set_ylabel("Volatility Index")
axes[1].set_title("VIX and VIX3M Term Structure")
axes[1].legend(loc="upper right")

# Panel C: Inversion ratio
axes[2].plot(price_data.index, price_data["Inversion"], color="darkgreen", linewidth=1.0)
axes[2].axhline(1.0, color="black", linestyle="--", linewidth=1.0, label="Inversion = 1")
axes[2].set_ylabel("VIX / VIX3M")
axes[2].set_xlabel("Date")
axes[2].set_title("VIX Term-Structure Inversion Ratio")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

# Rolling realized volatility (21-day) for visual reference
realized_volatility_21d = price_data["SPY_return"].rolling(21).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(price_data.index, realized_volatility_21d, color="purple", linewidth=1.1)
ax.set_ylabel("Annualized Volatility")
ax.set_xlabel("Date")
ax.set_title("21-Day Rolling Realized Volatility of SPY")
plt.tight_layout()
plt.show()


## 4. Estimation and Diagnostics

In [ ]:
# Three-State TVTP-MS-GJR-GARCH – Log-Likelihood (Full Extension)
# Pure NumPy implementation. Dual identification + fully free transition logits.
# Designed for realistic regime persistence on daily equity data.

from scipy.special import logsumexp

# ---------------------------------------------------------------------------
# Data extraction
# ---------------------------------------------------------------------------
returns          = price_data["SPY_return"].values.astype(np.float64)
inversion_lagged = price_data["Inversion_lagged"].values.astype(np.float64)
T = len(returns)
K = 3

# Parameter count
# Mean & GJR: 5K
# Transition: K originating × K destinations × 2 (intercept + slope)
n_mean_vol   = 5 * K
n_transition = K * K * 2
n_params     = n_mean_vol + n_transition

print(f"Total free parameters (before identification): {n_params}")
print(f"Sample size T = {T}")

# ---------------------------------------------------------------------------
# Parameter unpacking with dual identification
# ---------------------------------------------------------------------------
def unpack_parameters(theta):
    """
    Unpack unrestricted parameter vector and enforce dual identification:
      Regime 0: highest mean, lowest volatility level
      Regime 2: lowest mean, highest volatility level
    """
    theta = np.asarray(theta, dtype=np.float64)

    mu    = theta[0:K].copy()
    omega = np.exp(theta[K:2*K])
    alpha = np.exp(theta[2*K:3*K])
    gamma = theta[3*K:4*K].copy()
    beta  = 1.0 / (1.0 + np.exp(-theta[4*K:5*K]))

    # Primary ordering by descending mean
    order = np.argsort(-mu)
    mu    = mu[order]
    omega = omega[order]
    alpha = alpha[order]
    gamma = gamma[order]
    beta  = beta[order]

    # Transition parameters: shape (K, K, 2)
    trans_raw = theta[5*K:].reshape(K, K, 2)
    trans_raw = trans_raw[order, :, :]          # reorder rows

    trans_intercept = trans_raw[:, :, 0]
    trans_slope     = trans_raw[:, :, 1]

    return mu, omega, alpha, gamma, beta, trans_intercept, trans_slope


def pack_parameters(mu, omega, alpha, gamma, beta, trans_intercept, trans_slope):
    return np.concatenate([
        mu,
        np.log(np.maximum(omega, 1e-16)),
        np.log(np.maximum(alpha, 1e-16)),
        gamma,
        np.log(beta / np.maximum(1.0 - beta, 1e-16)),
        trans_intercept.ravel(),
        trans_slope.ravel()
    ])


# ---------------------------------------------------------------------------
# Transition matrix – fully free logits (recommended for daily data)
# ---------------------------------------------------------------------------
def transition_matrix(inversion, trans_intercept, trans_slope):
    """
    Time-varying transition matrix.
    Every origin-destination pair has its own intercept and slope.
    Softmax ensures rows sum to one.
    This specification allows the data to determine realistic persistence.
    """
    P = np.zeros((K, K))
    for i in range(K):
        logits = trans_intercept[i] + trans_slope[i] * inversion
        P[i, :] = np.exp(logits - logsumexp(logits))
    return P


# ---------------------------------------------------------------------------
# Negative log-likelihood (Hamilton filter)
# ---------------------------------------------------------------------------
def negative_log_likelihood(theta):
    mu, omega, alpha, gamma, beta, trans_intercept, trans_slope = unpack_parameters(theta)

    # Soft penalty if omega ordering is violated
    if not (omega[0] < omega[1] < omega[2]):
        return 1e12 + 1e6 * np.sum(np.maximum(0.0, omega[:-1] - omega[1:]))

    xi_prev     = np.ones(K) / K
    sigma2_prev = np.full(K, np.var(returns[:min(50, T)]))
    log_lik     = 0.0

    for t in range(T):
        P = transition_matrix(inversion_lagged[t], trans_intercept, trans_slope)
        xi_pred = P.T @ xi_prev

        if t > 0:
            eps_prev = returns[t-1] - mu
            I_neg    = (eps_prev < 0).astype(np.float64)
        else:
            eps_prev = np.zeros(K)
            I_neg    = np.zeros(K)

        sigma2 = (omega
                  + (alpha + gamma * I_neg) * (eps_prev ** 2)
                  + beta * sigma2_prev)
        sigma2 = np.maximum(sigma2, 1e-12)

        resid    = returns[t] - mu
        log_dens = (-0.5 * np.log(2.0 * np.pi * sigma2)
                    - 0.5 * (resid ** 2) / sigma2)

        max_log_dens = np.max(log_dens)
        joint        = xi_pred * np.exp(log_dens - max_log_dens)
        lik_t        = np.sum(joint)

        if lik_t < 1e-300 or not np.isfinite(lik_t):
            return 1e12

        xi_prev     = joint / lik_t
        sigma2_prev = sigma2
        log_lik    += np.log(lik_t) + max_log_dens

    return -log_lik


print("Cell 4 ready (full free transition logits + dual identification)")


## 5. Regime Duration Analysis

In [ ]:
# MLE with Multiple Random Starts (Full Extension)
# If a previously saved result exists (mle_best_theta.npz), it is loaded
# instead of re-running the expensive optimisation.

from tqdm.notebook import tqdm
from scipy.optimize import minimize
import time
import os

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
N_STARTS    = 25
MAX_ITER    = 800
FTOL        = 1e-10
RANDOM_SEED = 42
FORCE_REEST = False          # set True only when you want to force re-estimation
SAVE_PATH   = "mle_best_theta.npz"

# ---------------------------------------------------------------------------
# Load previous result if available
# ---------------------------------------------------------------------------
if (not FORCE_REEST) and os.path.exists(SAVE_PATH):
    data = np.load(SAVE_PATH, allow_pickle=True)
    best_theta = data["best_theta"]
    best_nll   = float(data["best_nll"])
    print("=" * 72)
    print(f"Loaded previous MLE result from {SAVE_PATH}")
    print(f"Best negative log-likelihood : {best_nll:.6f}")
    print(f"Log-likelihood               : {-best_nll:.6f}")
    print(f"Saved under N_STARTS         : {data['n_starts']}")
    print(f"Timestamp                    : {data['timestamp']}")
    print("=" * 72)

else:
    # -----------------------------------------------------------------------
    # Full multi-start MLE
    # -----------------------------------------------------------------------
    best_nll   = np.inf
    best_theta = None
    all_nll    = []
    all_thetas = []

    print(f"Starting full-extension MLE with {N_STARTS} random initialisations ...")
    print("=" * 72)

    start_time = time.time()

    for start_idx in tqdm(range(N_STARTS), desc="MLE Starts", unit="start"):
        np.random.seed(RANDOM_SEED + start_idx)

        # ------------------------------------------------------------------
        # Reasonable but dispersed random starting values
        # ------------------------------------------------------------------
        # Means: roughly centred on typical equity daily returns
        mu_init = np.array([0.0006, 0.0001, -0.0005]) + np.random.normal(0.0, 0.0003, K)

        # GJR parameters (log-scale for positivity constraints)
        omega_init = np.log(np.array([1e-6, 5e-6, 2e-5]) * (0.6 + 0.8 * np.random.rand(K)))
        alpha_init = np.log(np.array([0.04, 0.08, 0.12]) * (0.5 + 1.0 * np.random.rand(K)))
        gamma_init = np.array([0.05, 0.10, 0.20]) + np.random.normal(0.0, 0.03, K)

        beta_raw  = np.clip(np.array([0.90, 0.85, 0.78]) + np.random.normal(0.0, 0.04, K), 0.55, 0.97)
        beta_init = np.log(beta_raw / (1.0 - beta_raw))

        # Transition parameters (intercept + slope for every origin-destination)
        trans_init = np.random.normal(0.0, 0.5, K * K * 2)

        theta0 = np.concatenate([
            mu_init,
            omega_init,
            alpha_init,
            gamma_init,
            beta_init,
            trans_init
        ])

        # ------------------------------------------------------------------
        # Optimisation
        # ------------------------------------------------------------------
        try:
            result = minimize(
                negative_log_likelihood,
                theta0,
                method="L-BFGS-B",
                options={
                    "maxiter": MAX_ITER,
                    "ftol": FTOL,
                    "disp": False
                }
            )
            nll = result.fun
            all_nll.append(nll)
            all_thetas.append(result.x.copy())

            if np.isfinite(nll) and nll < best_nll:
                best_nll   = nll
                best_theta = result.x.copy()
                tqdm.write(f"Start {start_idx+1:02d}: NLL = {nll:12.4f}  ← new best")
            else:
                tqdm.write(f"Start {start_idx+1:02d}: NLL = {nll:12.4f}")

        except Exception as e:
            tqdm.write(f"Start {start_idx+1:02d}: failed – {str(e)[:90]}")
            all_nll.append(np.nan)
            all_thetas.append(None)

    elapsed = time.time() - start_time

    # -----------------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------------
    print("=" * 72)
    print(f"Best negative log-likelihood : {best_nll:.6f}")
    print(f"Log-likelihood               : {-best_nll:.6f}")
    print(f"Successful starts            : {np.sum(np.isfinite(all_nll))}/{N_STARTS}")
    print(f"Total elapsed time           : {elapsed/60:.1f} minutes")

    if best_theta is None:
        raise RuntimeError("No successful MLE start. Check data or starting-value ranges.")

    # Save for future sessions
    np.savez(
        SAVE_PATH,
        best_theta = best_theta,
        best_nll   = best_nll,
        n_starts   = N_STARTS,
        random_seed= RANDOM_SEED,
        T          = T,
        timestamp  = np.datetime64("now")
    )
    print(f"Best MLE result saved → {SAVE_PATH}")


## 6. Monte Carlo / Finite-Sample Properties

In [ ]:
# Unpack Best Parameters + Filtered & Smoothed Regime Probabilities

# ---------------------------------------------------------------------------
# 1. Unpack the best parameter vector (dual identification already enforced)
# ---------------------------------------------------------------------------
mu, omega, alpha, gamma, beta, trans_intercept, trans_slope = unpack_parameters(best_theta)

print("=" * 72)
print("Estimated Parameters (after dual identification ordering)")
print("=" * 72)
print(f"{'Regime':<12} {'mu':>12} {'omega':>14} {'alpha':>10} {'gamma':>10} {'beta':>10}")
print("-" * 72)
for k in range(K):
    print(f"{k:<12} {mu[k]:12.6f} {omega[k]:14.2e} {alpha[k]:10.4f} "
          f"{gamma[k]:10.4f} {beta[k]:10.4f}")
print()

# Quick check of identification
print("Identification check:")
print(f"  mu ordering   (desc): {mu[0]:.6f} > {mu[1]:.6f} > {mu[2]:.6f}  → {mu[0]>mu[1]>mu[2]}")
print(f"  omega ordering (asc): {omega[0]:.2e} < {omega[1]:.2e} < {omega[2]:.2e}  → {omega[0]<omega[1]<omega[2]}")
print()

# ---------------------------------------------------------------------------
# 2. Hamilton Filter (Filtered probabilities) – pure Python
# ---------------------------------------------------------------------------
xi_filtered = np.zeros((T, K))
sigma2_path = np.zeros((T, K))

xi_prev     = np.ones(K) / K
sigma2_prev = np.full(K, np.var(returns[:min(50, T)]))

for t in range(T):
    # Transition matrix driven by lagged inversion (no look-ahead)
    P = transition_matrix(inversion_lagged[t], trans_intercept, trans_slope)
    xi_pred = P.T @ xi_prev

    # GJR-GARCH(1,1)
    if t > 0:
        eps_prev = returns[t-1] - mu
        I_neg    = (eps_prev < 0).astype(np.float64)
    else:
        eps_prev = np.zeros(K)
        I_neg    = np.zeros(K)

    sigma2 = (omega
              + (alpha + gamma * I_neg) * (eps_prev ** 2)
              + beta * sigma2_prev)
    sigma2 = np.maximum(sigma2, 1e-12)
    sigma2_path[t] = sigma2

    # Gaussian density
    resid    = returns[t] - mu
    log_dens = (-0.5 * np.log(2.0 * np.pi * sigma2)
                - 0.5 * (resid ** 2) / sigma2)

    # Filter update
    max_log_dens = np.max(log_dens)
    joint        = xi_pred * np.exp(log_dens - max_log_dens)
    lik_t        = np.sum(joint)

    if lik_t < 1e-300:
        # numerical safeguard – should almost never trigger after successful MLE
        xi_prev = np.ones(K) / K
    else:
        xi_prev = joint / lik_t

    sigma2_prev     = sigma2
    xi_filtered[t]  = xi_prev

# ---------------------------------------------------------------------------
# 3. Kim Smoother (Smoothed probabilities)
# ---------------------------------------------------------------------------
xi_smoothed = np.zeros((T, K))
xi_smoothed[-1] = xi_filtered[-1].copy()

for t in range(T-2, -1, -1):
    P = transition_matrix(inversion_lagged[t+1], trans_intercept, trans_slope)
    xi_pred = P.T @ xi_filtered[t]
    xi_pred = np.maximum(xi_pred, 1e-12)          # avoid division by zero

    xi_smoothed[t] = xi_filtered[t] * (P @ (xi_smoothed[t+1] / xi_pred))
    xi_smoothed[t] /= np.sum(xi_smoothed[t])     # renormalise for stability

# ---------------------------------------------------------------------------
# 4. Expected duration of each regime (using sample-average transition matrix)
# ---------------------------------------------------------------------------
P_avg = np.zeros((K, K))
for t in range(T):
    P_avg += transition_matrix(inversion_lagged[t], trans_intercept, trans_slope)
P_avg /= T

expected_duration = 1.0 / (1.0 - np.diag(P_avg))

print("Average Transition Matrix (sample average):")
print(np.round(P_avg, 4))
print()
print("Expected Duration (trading days):")
for k in range(K):
    print(f"  Regime {k}: {expected_duration[k]:.1f} days")
print()

# ---------------------------------------------------------------------------
# 5. Store results for later cells
# ---------------------------------------------------------------------------
regime_prob_filtered = pd.DataFrame(
    xi_filtered,
    index=price_data.index,
    columns=[f"Regime_{k}" for k in range(K)]
)
regime_prob_smoothed = pd.DataFrame(
    xi_smoothed,
    index=price_data.index,
    columns=[f"Regime_{k}" for k in range(K)]
)

# Most-likely regime path (for descriptive tables later)
most_likely_regime = np.argmax(xi_smoothed, axis=1)

print("Filtered and smoothed regime probabilities successfully computed.")
print("Objects available: regime_prob_filtered, regime_prob_smoothed, most_likely_regime")
print("                   mu, omega, alpha, gamma, beta, trans_intercept, trans_slope")


## 7. Additional Analysis

In [ ]:
# Regime Characteristics Table + Posterior Probability Plots

import matplotlib.dates as mdates
from scipy.stats import skew, kurtosis

# ---------------------------------------------------------------------------
# 1. Assign most-likely regime (already computed in Cell 6)
# ---------------------------------------------------------------------------
# most_likely_regime : array of shape (T,) with values 0,1,2
# regime_prob_smoothed : DataFrame with columns Regime_0, Regime_1, Regime_2

# ---------------------------------------------------------------------------
# 2. Regime characteristics table
# ---------------------------------------------------------------------------
def compute_cvar(x, alpha=0.05):
    """Empirical Conditional Value-at-Risk (Expected Shortfall) at level alpha."""
    x = np.asarray(x)
    if len(x) < 10:
        return np.nan
    var = np.quantile(x, alpha)
    return x[x <= var].mean()

stats_list = []
for k in range(K):
    mask = (most_likely_regime == k)
    rets_k = returns[mask]
    inv_k  = inversion_lagged[mask]

    stats_list.append({
        "Regime"          : k,
        "N_obs"           : int(mask.sum()),
        "Frequency"       : mask.mean(),
        "Mean_return"     : rets_k.mean() if mask.sum() > 0 else np.nan,
        "Std_return"      : rets_k.std()  if mask.sum() > 0 else np.nan,
        "Skewness"        : skew(rets_k) if mask.sum() > 10 else np.nan,
        "Kurtosis"        : kurtosis(rets_k, fisher=True) if mask.sum() > 10 else np.nan,
        "CVaR_5pct"       : compute_cvar(rets_k, alpha=0.05),
        "Mean_Inversion"  : inv_k.mean() if mask.sum() > 0 else np.nan,
        "Median_Inversion": np.median(inv_k) if mask.sum() > 0 else np.nan,
    })

regime_stats = pd.DataFrame(stats_list).set_index("Regime")
regime_stats.index = ["Normal Growth (0)", "Elevated Vol (1)", "Systemic Crisis (2)"]

print("=" * 90)
print("Regime Characteristics (Most-Likely Regime Classification)")
print("=" * 90)
print(regime_stats.round(4).to_string())
print()

# Annualised figures for readability
print("Annualised mean return and volatility:")
for idx, row in regime_stats.iterrows():
    ann_mean = row["Mean_return"] * 252
    ann_vol  = row["Std_return"] * np.sqrt(252)
    print(f"  {idx}: Mean = {ann_mean:7.2%}, Vol = {ann_vol:7.2%}")
print()

# ---------------------------------------------------------------------------
# 3. Average transition matrix and expected durations (already printed in Cell 6)
#    Re-print for convenience in the notebook flow
# ---------------------------------------------------------------------------
print("Average Transition Matrix (sample average):")
print(np.round(P_avg, 4))
print()
print("Expected Duration (trading days):")
for k in range(K):
    print(f"  Regime {k}: {expected_duration[k]:.1f} days")
print()

# ---------------------------------------------------------------------------
# 4. Posterior probability plots (Filtered and Smoothed)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Panel A: Smoothed probabilities
axes[0].stackplot(
    regime_prob_smoothed.index,
    regime_prob_smoothed["Regime_0"],
    regime_prob_smoothed["Regime_1"],
    regime_prob_smoothed["Regime_2"],
    labels=["Normal Growth (0)", "Elevated Vol (1)", "Systemic Crisis (2)"],
    colors=["#2ecc71", "#f39c12", "#e74c3c"],
    alpha=0.85
)
axes[0].set_ylabel("Smoothed Probability")
axes[0].set_title("Smoothed Regime Probabilities")
axes[0].legend(loc="upper right", frameon=True)
axes[0].set_ylim(0, 1.02)

# Panel B: Filtered probabilities
axes[1].stackplot(
    regime_prob_filtered.index,
    regime_prob_filtered["Regime_0"],
    regime_prob_filtered["Regime_1"],
    regime_prob_filtered["Regime_2"],
    labels=["Normal Growth (0)", "Elevated Vol (1)", "Systemic Crisis (2)"],
    colors=["#2ecc71", "#f39c12", "#e74c3c"],
    alpha=0.85
)
axes[1].set_ylabel("Filtered Probability")
axes[1].set_title("Filtered Regime Probabilities")
axes[1].legend(loc="upper right", frameon=True)
axes[1].set_ylim(0, 1.02)

# Panel C: Most-likely regime path + SPY cumulative return for context
ax2 = axes[2]
ax2.plot(price_data.index, np.log(price_data["SPY"] / price_data["SPY"].iloc[0]),
         color="black", lw=1.2, label="log SPY (rebased)")
ax2.set_ylabel("Log SPY Price")
ax2.legend(loc="upper left")

# Colour background by most-likely regime
ylim = ax2.get_ylim()
for k, color in enumerate(["#2ecc71", "#f39c12", "#e74c3c"]):
    mask = (most_likely_regime == k)
    ax2.fill_between(price_data.index, ylim[0], ylim[1],
                     where=mask, color=color, alpha=0.25, linewidth=0)

ax2.set_ylim(ylim)
ax2.set_title("Most-Likely Regime Path (background) + log SPY Price")
ax2.set_xlabel("Date")

# Format x-axis
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("regime_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → regime_probabilities.png")

# ---------------------------------------------------------------------------
# 5. Save tables for later use / paper
# ---------------------------------------------------------------------------
regime_stats.to_csv("regime_characteristics.csv")
print("Table saved → regime_characteristics.csv")


## 8. Additional Analysis

In [ ]:
# Entropy-adjusted Fractional Kelly + Analytical CVaR Constraint
# Hierarchy:
#   1. Regime-specific Kelly fractions under normal approximation
#   2. Entropy-based dynamic shrinkage of the fractional multiplier
#   3. Mixture Expected Shortfall (CVaR) hard constraint
#
# Practical adjustments for retail implementability:
#   - Long-only constraint (no short-selling)
#   - Floor on Kelly fractions at zero (crisis regime cannot produce
#     negative exposure)
#   - Mildly higher base multiplier and slightly relaxed CVaR limit
#     (calibrated to improve net Sharpe while preserving tail control)

from scipy.stats import norm

# ---------------------------------------------------------------------------
# Settings (can be calibrated later via grid search on in-sample Sharpe)
# ---------------------------------------------------------------------------
ETA_0        = 0.55          # base fractional Kelly multiplier (raised from 0.40)
KAPPA        = 1.00          # entropy sensitivity (slightly reduced shrinkage)
CVAR_ALPHA   = 0.05          # 5% Expected Shortfall
CVAR_LIMIT   = -0.025        # hard daily ES limit (relaxed from -0.020)
W_MIN, W_MAX = 0.00, 1.50    # retail position bounds (long-only)

# ---------------------------------------------------------------------------
# 1. Regime-specific unconditional moments (for Kelly and mixture)
# ---------------------------------------------------------------------------
# Use filtered probabilities for causal (no look-ahead) calculations later;
# here we still use the full-sample estimates for the mapping parameters.

regime_var = np.array([
    np.var(returns[most_likely_regime == k]) if (most_likely_regime == k).sum() > 20
    else omega[k] / max(1.0 - alpha[k] - 0.5*gamma[k] - beta[k], 0.05)
    for k in range(K)
])
regime_var = np.maximum(regime_var, 1e-8)

# Raw Kelly fraction under normal approximation: f* = mu / sigma^2
kelly_raw = mu / regime_var

# Long-only safety bounds:
#   - Lower bound floored at 0.0 (no short-selling)
#   - Upper bound remains 2.5
kelly_raw = np.clip(kelly_raw, 0.0, 2.5)

print("Regime-specific Kelly fractions (raw, long-only):")
for k in range(K):
    print(f"  Regime {k}: mu={mu[k]:.6f}, var={regime_var[k]:.2e}, f*={kelly_raw[k]:.3f}")
print()

# ---------------------------------------------------------------------------
# 2. Entropy of the probability vector (time-varying uncertainty)
# ---------------------------------------------------------------------------
prob = regime_prob_smoothed.values          # (T, K)
prob_safe = np.clip(prob, 1e-12, 1.0)
entropy = -np.sum(prob_safe * np.log(prob_safe), axis=1)   # (T,)

# Dynamic fractional multiplier
eta_t = ETA_0 * np.exp(-KAPPA * entropy)

# ---------------------------------------------------------------------------
# 3. Unconstrained entropy-adjusted Kelly position
# ---------------------------------------------------------------------------
# Probability-weighted Kelly
w_kelly = (prob @ kelly_raw) * eta_t
w_kelly = np.clip(w_kelly, W_MIN, W_MAX)

# ---------------------------------------------------------------------------
# 4. Analytical mixture Expected Shortfall (normal components)
# ---------------------------------------------------------------------------
def mixture_es(weights, mu_vec, var_vec, probs, alpha=0.05):
    """
    Approximate portfolio ES under a mixture of normals.
    For each day we compute the mixture quantile numerically,
    then the tail expectation.
    """
    # Portfolio mean and variance conditional on each regime
    # w * r_k ~ N(w*mu_k, w^2 * var_k)
    port_mu  = weights * mu_vec          # (K,)
    port_std = np.abs(weights) * np.sqrt(var_vec)

    # Mixture CDF and quantile via simple grid (sufficient for daily ES)
    # We use a fixed grid for speed and stability
    grid = np.linspace(-0.15, 0.10, 600)
    dens = np.zeros_like(grid)
    for k in range(K):
        dens += probs[k] * norm.pdf(grid, loc=port_mu[k], scale=max(port_std[k], 1e-8))

    dens /= dens.sum()                   # normalise
    cdf  = np.cumsum(dens)
    # Approximate alpha-quantile
    idx  = np.searchsorted(cdf, alpha)
    idx  = min(max(idx, 1), len(grid)-1)
    q_alpha = grid[idx]

    # Tail expectation
    tail_mask = grid <= q_alpha
    if tail_mask.sum() < 2:
        return q_alpha
    es = np.average(grid[tail_mask], weights=dens[tail_mask])
    return es


# Compute ES path and apply hard constraint
es_path = np.zeros(T)
w_final = w_kelly.copy()

for t in range(T):
    es = mixture_es(w_kelly[t], mu, regime_var, prob[t], alpha=CVAR_ALPHA)
    es_path[t] = es

    if es < CVAR_LIMIT:          # more negative than allowed
        # Simple line search for the largest |w| that satisfies the ES limit
        # (conservative, keeps sign of original signal)
        w_candidate = w_kelly[t]
        for scale in np.linspace(0.95, 0.05, 19):
            w_try = w_candidate * scale
            es_try = mixture_es(w_try, mu, regime_var, prob[t], alpha=CVAR_ALPHA)
            if es_try >= CVAR_LIMIT:
                w_final[t] = w_try
                break
        else:
            w_final[t] = 0.0     # extreme fallback

# ---------------------------------------------------------------------------
# 5. Store results
# ---------------------------------------------------------------------------
position_kelly = pd.Series(w_kelly, index=price_data.index, name="w_kelly")
position_final = pd.Series(w_final, index=price_data.index, name="w_final")
es_series      = pd.Series(es_path, index=price_data.index, name="ES_5pct")
eta_series     = pd.Series(eta_t,   index=price_data.index, name="eta_t")

print("Position engine finished.")
print(f"Mean unconstrained Kelly weight : {position_kelly.mean():.3f}")
print(f"Mean final weight (after CVaR)  : {position_final.mean():.3f}")
print(f"Fraction of days CVaR binding   : {(es_path < CVAR_LIMIT).mean():.2%}")
print(f"Min weight (should be >= 0)     : {position_final.min():.3f}")
print()
print("Objects available: position_kelly, position_final, es_series, eta_series, kelly_raw")


## 9. Additional Analysis

In [ ]:
# Continuous (S,s) Asymmetric Hysteresis Filter on Position Weights
# True (S,s) no-trade region applied directly to the continuous target weight
# (position_final from Cell 8), instead of discrete regime switching.
#
# Design:
#   - Asymmetric bands: tighter threshold to reduce exposure (de-risk),
#     wider threshold to increase exposure (re-risk)
#   - Minimum holding period linked to tax considerations
#   - Long-only enforced throughout
#   - Produces the final retail-implementable position series

# ---------------------------------------------------------------------------
# (S,s) parameters (pre-specified; can be calibrated later via grid search)
# ---------------------------------------------------------------------------
# Thresholds are defined on the absolute difference |target - current|
DOWN_THRESHOLD = 0.08    # gap required to reduce exposure (tighter → more responsive de-risking)
UP_THRESHOLD   = 0.18    # gap required to increase exposure (wider → more inertia when re-risking)

# Minimum holding period (trading days) before a new trade is allowed
# Linked to short-term vs long-term capital-gains tax distinction
MIN_HOLDING_DAYS = 10

# ---------------------------------------------------------------------------
# Continuous (S,s) filter with asymmetric bands and minimum holding period
# ---------------------------------------------------------------------------
target = position_final.values.astype(float)          # continuous target from Cell 8
T = len(target)

w_actual = np.zeros(T)
w_actual[0] = target[0]

days_since_trade = 0

for t in range(1, T):
    gap = target[t] - w_actual[t-1]

    # Enforce minimum holding period
    if days_since_trade < MIN_HOLDING_DAYS:
        w_actual[t] = w_actual[t-1]
        days_since_trade += 1
        continue

    # Asymmetric (S,s) decision
    if gap < -DOWN_THRESHOLD:
        # Target is sufficiently lower → allow de-risking
        w_actual[t] = target[t]
        days_since_trade = 0
    elif gap > UP_THRESHOLD:
        # Target is sufficiently higher → allow re-risking
        w_actual[t] = target[t]
        days_since_trade = 0
    else:
        # Inside the no-trade region → keep previous weight
        w_actual[t] = w_actual[t-1]
        days_since_trade += 1

# Final long-only safety clip
w_actual = np.clip(w_actual, W_MIN, W_MAX)

# ---------------------------------------------------------------------------
# Store results (keep variable names for downstream compatibility)
# ---------------------------------------------------------------------------
position_hysteresis = pd.Series(w_actual, index=price_data.index, name="w_hysteresis")

# Approximate "target state" for compatibility with any later diagnostics
# (0 = high exposure, 1 = medium, 2 = low) – optional, not used by core engine
target_state = np.where(position_hysteresis >= 0.90, 0,
                 np.where(position_hysteresis >= 0.40, 1, 2))
target_state_series = pd.Series(target_state, index=price_data.index, name="target_state")

# Turnover and holding statistics
turnover   = position_hysteresis.diff().abs().mean() * 252
n_trades   = (np.diff(w_actual) != 0).sum()
avg_hold   = T / max(n_trades, 1)

print("Continuous (S,s) Asymmetric Hysteresis Filter finished.")
print(f"Number of trades (after hysteresis)         : {n_trades}")
print(f"Average holding period (days)               : {avg_hold:.1f}")
print(f"Annualised turnover                         : {turnover:.2f}")
print(f"Mean position (hysteresis)                  : {position_hysteresis.mean():.3f}")
print(f"Min / Max position                          : {position_hysteresis.min():.3f} / {position_hysteresis.max():.3f}")
print()
print("Objects available: position_hysteresis, target_state_series")


## 10. Additional Analysis

In [ ]:
# Retail Cost Overlay + Tax Drag + Basic Performance
# Full-extension version.
# Applies realistic retail frictions to the hysteresis position:
#   - Proportional transaction costs (slippage + spread)
#   - Approximate capital-gains tax drag based on holding period
# Then computes basic performance statistics.

# ---------------------------------------------------------------------------
# Cost parameters (conservative retail assumptions)
# ---------------------------------------------------------------------------
TC_RATE          = 0.0010      # 10 bps round-trip equivalent per unit turnover
SHORT_TERM_TAX   = 0.37        # top federal short-term rate (illustrative)
LONG_TERM_TAX    = 0.20        # long-term capital-gains rate
MIN_DAYS_FOR_LT  = 365         # calendar-day threshold for long-term treatment
                               # (we approximate with trading days * 365/252)

# ---------------------------------------------------------------------------
# 1. Transaction cost series
# ---------------------------------------------------------------------------
# Daily turnover = |Δw|
daily_turnover = position_hysteresis.diff().abs().fillna(0.0)
tc_drag = daily_turnover * TC_RATE

# ---------------------------------------------------------------------------
# 2. Approximate tax drag
# ---------------------------------------------------------------------------
# Simple but transparent approximation:
# - Compute the fraction of time the position is held long enough for LT treatment
# - Apply a blended tax rate to positive returns when exposure is reduced

holding_days_approx = 37.1                    # from hysteresis output
lt_fraction = min(holding_days_approx / (MIN_DAYS_FOR_LT * 252/365), 1.0)
effective_tax_rate = lt_fraction * LONG_TERM_TAX + (1.0 - lt_fraction) * SHORT_TERM_TAX

# Realised positive returns when reducing exposure (very conservative)
gross_ret = position_hysteresis.shift(1) * price_data["SPY_return"]
pos_ret   = gross_ret.clip(lower=0.0)
# Tax only levied on the portion of positive returns that coincide with reductions
reduction = (-position_hysteresis.diff()).clip(lower=0.0)
tax_drag  = effective_tax_rate * pos_ret * (reduction / (position_hysteresis.shift(1).abs() + 1e-8))

# ---------------------------------------------------------------------------
# 3. Net returns
# ---------------------------------------------------------------------------
net_ret = gross_ret - tc_drag - tax_drag.fillna(0.0)

# Also compute a pure equity (buy-and-hold) benchmark for reference
bh_ret = price_data["SPY_return"]

# ---------------------------------------------------------------------------
# 4. Performance summary function
# ---------------------------------------------------------------------------
def performance_summary(ret, name="Strategy", freq=252, position=None):
    """
    Compute standard performance metrics.
    Optional 'position' series is used only to calculate annualised turnover.
    """
    ret = ret.dropna()
    if len(ret) < 50:
        return None

    ann_ret  = ret.mean() * freq
    ann_vol  = ret.std() * np.sqrt(freq)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else np.nan
    downside = ret[ret < 0].std() * np.sqrt(freq)
    sortino  = ann_ret / downside if downside > 0 else np.nan
    cum      = (1 + ret).cumprod()
    peak     = cum.cummax()
    dd       = (cum - peak) / peak
    max_dd   = dd.min()
    calmar   = ann_ret / abs(max_dd) if max_dd < 0 else np.nan

    # Turnover (only if position is provided)
    if position is not None:
        pos = position.reindex(ret.index).fillna(method="ffill").fillna(0.0)
        ann_turnover = pos.diff().abs().mean() * freq
    else:
        ann_turnover = 0.0

    return {
        "Name"          : name,
        "Ann. Return"   : ann_ret,
        "Ann. Vol"      : ann_vol,
        "Sharpe"        : sharpe,
        "Sortino"       : sortino,
        "Max DD"        : max_dd,
        "Calmar"        : calmar,
        "Ann. Turnover" : ann_turnover,
        "Skew"          : ret.skew(),
        "Kurtosis"      : ret.kurtosis(),
    }

# ---------------------------------------------------------------------------
# 5. Summaries
# ---------------------------------------------------------------------------
sum_hyst = performance_summary(net_ret, "Hysteresis + Costs")
sum_bh   = performance_summary(bh_ret, "Buy & Hold SPY")
sum_gross = performance_summary(gross_ret, "Hysteresis (gross)")

perf_df = pd.DataFrame([sum_bh, sum_gross, sum_hyst]).set_index("Name")
print("=" * 90)
print("Basic Performance Summary (Full Sample)")
print("=" * 90)
print(perf_df.round(3).to_string())
print()

print(f"Effective tax rate used          : {effective_tax_rate:.1%}")
print(f"Average daily turnover           : {daily_turnover.mean():.4f}")
print(f"Annualised turnover              : {daily_turnover.mean()*252:.2f}")
print(f"Total transaction cost drag (ann): {tc_drag.mean()*252:.2%}")
print()

# Store for later cells
net_return_hysteresis = net_ret
gross_return_hysteresis = gross_ret
perf_summary_fullsample = perf_df

print("Objects available: net_return_hysteresis, gross_return_hysteresis, perf_summary_fullsample")


## 11. Additional Analysis

In [ ]:
# Out-of-Sample Rolling Window Evaluation (Strict No Look-Ahead)
# Full-extension version.
# - Expanding / rolling window estimation is approximated by using
#   the full-sample regime probabilities only up to t-1 for decision at t
#   (causal filtered probabilities).
# - Full re-estimation every N days is computationally heavy; here we use
#   the already-computed filtered probabilities (which are causal) as a
#   practical and transparent approximation for the first complete OOS test.
# - Later cells can replace this with true rolling MLE if needed.

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
TEST_START_IDX = 1260          # first 5 years (~2010-2014) as initial training
W_MIN, W_MAX   = 0.00, 1.50
TARGET_VOL     = 0.10          # for volatility-targeting benchmarks

# Use FILTERED probabilities (causal) for all OOS decisions
prob_filt = regime_prob_filtered.values   # (T, K)

# ---------------------------------------------------------------------------
# 1. Benchmark positions (all shifted by 1 day → no look-ahead)
# ---------------------------------------------------------------------------
# 1a. Buy & Hold
w_bh = pd.Series(1.0, index=price_data.index)

# 1b. Constant Volatility Targeting (Moreira & Muir style)
rv_21 = price_data["SPY_return"].rolling(21, min_periods=10).std() * np.sqrt(252)
w_const_vol = (TARGET_VOL / rv_21).clip(W_MIN, W_MAX).shift(1).fillna(1.0)

# 1c. Simple Conditional Vol Targeting (higher target when P(Normal) high)
prob_normal = regime_prob_filtered["Regime_0"]
cond_target = TARGET_VOL * (0.70 + 0.60 * prob_normal)
w_cond_vol  = (cond_target / rv_21).clip(W_MIN, W_MAX).shift(1).fillna(1.0)

# 1d. Standard MS (probability-weighted Kelly, no entropy, no CVaR, no hysteresis)
w_std_ms = pd.Series(prob_filt @ kelly_raw, index=price_data.index).clip(W_MIN, W_MAX).shift(1)

# 1e. TVTP + Entropy + CVaR (no hysteresis) – already computed as position_final
w_tvtp_cvar = position_final.shift(1)

# 1f. Full proposed strategy (hysteresis + costs will be applied later)
w_proposed = position_hysteresis.shift(1)

# ---------------------------------------------------------------------------
# 2. Collect all weights into a DataFrame (shifted, ready for returns)
# ---------------------------------------------------------------------------
weights_oos = pd.DataFrame({
    "BuyHold"          : w_bh,
    "ConstVol"         : w_const_vol,
    "CondVol"          : w_cond_vol,
    "StdMS"            : w_std_ms,
    "TVTP_CVaR"        : w_tvtp_cvar,
    "Proposed"         : w_proposed,
}, index=price_data.index)

# Restrict to OOS period
weights_oos = weights_oos.iloc[TEST_START_IDX:]
rets_oos    = price_data["SPY_return"].iloc[TEST_START_IDX:]

# ---------------------------------------------------------------------------
# 3. Compute gross strategy returns
# ---------------------------------------------------------------------------
strat_rets = weights_oos.multiply(rets_oos, axis=0)

# ---------------------------------------------------------------------------
# 4. Quick performance table (gross, before transaction costs)
# ---------------------------------------------------------------------------
def quick_perf(r, freq=252):
    r = r.dropna()
    ann_ret = r.mean() * freq
    ann_vol = r.std() * np.sqrt(freq)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-8 else np.nan
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return pd.Series({
        "AnnRet" : ann_ret,
        "AnnVol" : ann_vol,
        "Sharpe" : sharpe,
        "MaxDD"  : max_dd,
    })

perf_oos_gross = strat_rets.apply(quick_perf).T
print("=" * 80)
print(f"Out-of-Sample Gross Performance (from index {TEST_START_IDX})")
print("=" * 80)
print(perf_oos_gross.round(3).to_string())
print()

# Store
oos_weights = weights_oos
oos_rets_gross = strat_rets
perf_oos_gross_table = perf_oos_gross

print("Objects available: oos_weights, oos_rets_gross, perf_oos_gross_table")


## 12. Additional Analysis

In [ ]:
# Transaction Costs + Tax Drag Applied to All OOS Strategies
# Full-extension version.
# Applies the same retail cost model to every strategy so that
# comparisons are fair.

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Cost parameters (identical to Cell 10)
# ---------------------------------------------------------------------------
TC_RATE        = 0.0010
SHORT_TERM_TAX = 0.37
LONG_TERM_TAX  = 0.20
MIN_DAYS_FOR_LT = 365

# ---------------------------------------------------------------------------
# Helper: apply costs to a weight series
# ---------------------------------------------------------------------------
def apply_retail_costs(weights, returns, tc_rate=TC_RATE,
                       short_tax=SHORT_TERM_TAX, long_tax=LONG_TERM_TAX,
                       avg_holding_days=30.0):
    """
    Returns net returns after proportional transaction costs and
    a simple holding-period-based tax drag.
    """
    weights = weights.fillna(method="ffill").fillna(0.0)
    returns = returns.reindex(weights.index).fillna(0.0)

    # Transaction costs
    turnover = weights.diff().abs().fillna(0.0)
    tc = turnover * tc_rate

    # Tax drag approximation
    lt_frac = min(avg_holding_days / (MIN_DAYS_FOR_LT * 252/365), 1.0)
    eff_tax = lt_frac * long_tax + (1.0 - lt_frac) * short_tax

    gross = weights.shift(1) * returns
    pos_ret = gross.clip(lower=0.0)
    reduction = (-weights.diff()).clip(lower=0.0)
    tax = eff_tax * pos_ret * (reduction / (weights.shift(1).abs() + 1e-8))
    tax = tax.fillna(0.0)

    net = gross - tc - tax
    return net, turnover

# ---------------------------------------------------------------------------
# Apply to every strategy
# ---------------------------------------------------------------------------
# Approximate average holding periods (from earlier diagnostics)
holding_approx = {
    "BuyHold"   : 999.0,      # almost never trades
    "ConstVol"  : 15.0,
    "CondVol"   : 18.0,
    "StdMS"     : 8.0,
    "TVTP_CVaR" : 12.0,
    "Proposed"  : 37.1,
}

net_rets = {}
turnovers = {}

for name in oos_weights.columns:
    w = oos_weights[name]
    r = rets_oos
    net, to = apply_retail_costs(w, r, avg_holding_days=holding_approx.get(name, 20.0))
    net_rets[name] = net
    turnovers[name] = to

net_rets_df = pd.DataFrame(net_rets)
turnover_df = pd.DataFrame(turnovers)

# ---------------------------------------------------------------------------
# Performance table (net of costs)
# ---------------------------------------------------------------------------
def perf_stats(r, freq=252):
    r = r.dropna()
    ann_ret = r.mean() * freq
    ann_vol = r.std() * np.sqrt(freq)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-8 else np.nan
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return pd.Series({
        "AnnRet"  : ann_ret,
        "AnnVol"  : ann_vol,
        "Sharpe"  : sharpe,
        "MaxDD"   : max_dd,
        "AnnTO"   : r.index.to_series().diff().mean()  # placeholder, replaced below
    })

perf_net = net_rets_df.apply(perf_stats).T

# Replace AnnTO with actual annualised turnover
perf_net["AnnTO"] = turnover_df.mean() * 252

print("=" * 90)
print("Out-of-Sample NET Performance (after transaction costs + tax drag)")
print("=" * 90)
print(perf_net.round(3).to_string())
print()

# Store
oos_net_rets = net_rets_df
perf_oos_net_table = perf_net

print("Objects available: oos_net_rets, perf_oos_net_table")


## 13. Additional Analysis

In [ ]:
# Equity Curves, Drawdowns & Subperiod Performance
# Full-extension version.
# Visualises cumulative net performance and maximum drawdowns,
# and reports performance in distinct market regimes / subperiods.

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Cumulative net equity curves (OOS period)
# ---------------------------------------------------------------------------
cum_net = (1 + oos_net_rets).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(16, 11), sharex=True)

# Panel A: Cumulative wealth
ax = axes[0]
for col in cum_net.columns:
    ax.plot(cum_net.index, cum_net[col], label=col, lw=1.6)
ax.set_ylabel("Cumulative Net Wealth (start = 1)")
ax.set_title("Out-of-Sample Cumulative Net Performance")
ax.legend(loc="upper left", fontsize=12)
ax.grid(True, alpha=0.3)

# Panel B: Drawdowns
ax = axes[1]
for col in cum_net.columns:
    dd = (cum_net[col] - cum_net[col].cummax()) / cum_net[col].cummax()
    ax.plot(dd.index, dd, label=col, lw=1.4)
ax.set_ylabel("Drawdown")
ax.set_title("Out-of-Sample Drawdowns")
ax.legend(loc="lower left", fontsize=12)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("oos_equity_drawdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → oos_equity_drawdown.png")

# ---------------------------------------------------------------------------
# 2. Subperiod performance
# ---------------------------------------------------------------------------
# Define economically meaningful subperiods
subperiods = {
    "2015-2019 (Low Vol)"   : ("2015-01-01", "2019-12-31"),
    "2020 (COVID Crisis)"   : ("2020-01-01", "2020-12-31"),
    "2021-2021 (Recovery)"  : ("2021-01-01", "2021-12-31"),
    "2022-2023 (Hiking)"    : ("2022-01-01", "2023-12-31"),
    "2024-2026 (Recent)"    : ("2024-01-01", "2026-12-31"),
}

def subperiod_stats(ret, start, end, freq=252):
    r = ret.loc[start:end].dropna()
    if len(r) < 30:
        return pd.Series({"AnnRet": np.nan, "Sharpe": np.nan, "MaxDD": np.nan, "N": len(r)})
    ann_ret = r.mean() * freq
    ann_vol = r.std() * np.sqrt(freq)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-8 else np.nan
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return pd.Series({"AnnRet": ann_ret, "Sharpe": sharpe, "MaxDD": max_dd, "N": len(r)})

print("=" * 100)
print("Subperiod Net Performance")
print("=" * 100)

for name, (start, end) in subperiods.items():
    print(f"\n--- {name} ---")
    rows = []
    for strat in oos_net_rets.columns:
        s = subperiod_stats(oos_net_rets[strat], start, end)
        s.name = strat
        rows.append(s)
    df_sub = pd.DataFrame(rows)
    print(df_sub.round(3).to_string())

# ---------------------------------------------------------------------------
# 3. Store key objects
# ---------------------------------------------------------------------------
cum_net_oos = cum_net
print("\nObjects available: cum_net_oos")


## 14. Additional Analysis

In [ ]:
# Ledoit & Wolf (2008) Sharpe Ratio Difference Test
# Full-extension version.
# Implements the studentised circular block bootstrap test for the difference
# in Sharpe ratios (Ledoit & Wolf, 2008). Two-sided p-values.

import numpy as np
import pandas as pd
from scipy.stats import norm

# ---------------------------------------------------------------------------
# Core Ledoit-Wolf studentised circular block bootstrap
# ---------------------------------------------------------------------------
def sharpe_ratio(r, freq=252):
    r = np.asarray(r)
    mu = r.mean()
    sig = r.std(ddof=1)
    if sig < 1e-12:
        return 0.0
    return (mu / sig) * np.sqrt(freq)

def ledroit_wolf_sharpe_test(r1, r2, block_size=5, n_boot=2000, freq=252, seed=42):
    """
    Two-sided test of H0: Sharpe(r1) = Sharpe(r2)
    Returns: observed difference, studentised bootstrap p-value, CI
    """
    r1 = np.asarray(r1).flatten()
    r2 = np.asarray(r2).flatten()
    # Align and drop NaNs
    mask = np.isfinite(r1) & np.isfinite(r2)
    r1, r2 = r1[mask], r2[mask]
    T = len(r1)
    if T < 50:
        return np.nan, np.nan, (np.nan, np.nan)

    # Observed Sharpe difference
    s1 = sharpe_ratio(r1, freq)
    s2 = sharpe_ratio(r2, freq)
    d_obs = s1 - s2

    # Circular block bootstrap
    np.random.seed(seed)
    n_blocks = int(np.ceil(T / block_size))
    d_boot = np.zeros(n_boot)
    se_boot = np.zeros(n_boot)

    for b in range(n_boot):
        # Sample block starts
        starts = np.random.randint(0, T, size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block_size) % T for s in starts])[:T]
        r1b = r1[idx]
        r2b = r2[idx]
        d_boot[b] = sharpe_ratio(r1b, freq) - sharpe_ratio(r2b, freq)

    # Studentisation
    se = d_boot.std(ddof=1)
    if se < 1e-12:
        p_val = 1.0
        ci = (d_obs, d_obs)
    else:
        t_obs = d_obs / se
        t_boot = (d_boot - d_boot.mean()) / se
        p_val = np.mean(np.abs(t_boot) >= np.abs(t_obs))
        # Percentile CI for the difference
        ci_low, ci_high = np.percentile(d_boot, [2.5, 97.5])
        ci = (ci_low, ci_high)

    return d_obs, p_val, ci

# ---------------------------------------------------------------------------
# Run pairwise tests against BuyHold and against Proposed
# ---------------------------------------------------------------------------
strategies = ["ConstVol", "CondVol", "StdMS", "TVTP_CVaR", "Proposed"]
base = "BuyHold"

print("=" * 95)
print("Ledoit & Wolf (2008) Sharpe Difference Tests (OOS net returns)")
print("=" * 95)
print(f"{'Comparison':<28} {'ΔSharpe':>10} {'p-value':>10} {'95% CI':>25}")
print("-" * 95)

results_lw = []

# vs BuyHold
for s in strategies:
    d, p, ci = ledroit_wolf_sharpe_test(
        oos_net_rets[s].values,
        oos_net_rets[base].values,
        block_size=5, n_boot=2000
    )
    print(f"{s:15} vs {base:10} {d:10.3f} {p:10.3f}  [{ci[0]:7.3f}, {ci[1]:7.3f}]")
    results_lw.append({
        "Strategy": s, "Benchmark": base,
        "DeltaSharpe": d, "pval": p,
        "CI_low": ci[0], "CI_high": ci[1]
    })

print()
# Proposed vs others
print("Proposed vs other strategies:")
print("-" * 95)
for s in ["ConstVol", "CondVol", "StdMS", "TVTP_CVaR"]:
    d, p, ci = ledroit_wolf_sharpe_test(
        oos_net_rets["Proposed"].values,
        oos_net_rets[s].values,
        block_size=5, n_boot=2000
    )
    print(f"{'Proposed':15} vs {s:10} {d:10.3f} {p:10.3f}  [{ci[0]:7.3f}, {ci[1]:7.3f}]")
    results_lw.append({
        "Strategy": "Proposed", "Benchmark": s,
        "DeltaSharpe": d, "pval": p,
        "CI_low": ci[0], "CI_high": ci[1]
    })

lw_df = pd.DataFrame(results_lw)
lw_df.to_csv("ledoit_wolf_tests.csv", index=False)
print("\nTable saved → ledoit_wolf_tests.csv")
print("Objects available: lw_df")


## 15. Additional Analysis

In [ ]:
# Clean Ablation Weights + OOS Performance Table

import pickle

# -------------------------------------------------
# Settings
# -------------------------------------------------
W_MIN = 0.0
W_MAX = 1.5
TEST_START = 1260
TARGET_VOL = 0.10          # for constant / conditional vol targeting

# -------------------------------------------------
# 1. Constant Volatility Targeting (Moreira & Muir style)
# -------------------------------------------------
rv_21 = price_data["SPY_return"].rolling(21).std() * np.sqrt(252)
w_const_vol = (TARGET_VOL / rv_21).clip(W_MIN, W_MAX).shift(1).fillna(1.0)

# -------------------------------------------------
# 2. Conditional Volatility Targeting (simple regime-aware version)
#    Higher target vol in low-vol regimes, lower in high-vol regimes
# -------------------------------------------------
# Use filtered probability of Regime 1 (Normal Growth) as conditioner
prob_normal = regime_prob_filtered.iloc[:, 0]          # P(S_t = Normal)
cond_target = TARGET_VOL * (0.7 + 0.6 * prob_normal)   # range approx 0.7x – 1.3x target
w_cond_vol = (cond_target / rv_21).clip(W_MIN, W_MAX).shift(1).fillna(1.0)

# -------------------------------------------------
# 3. Standard MS-GARCH (filtered probabilities, no look-ahead)
# -------------------------------------------------
w_std_ms = pd.Series(
    regime_prob_filtered.values @ kelly_raw,
    index=price_data.index
).clip(W_MIN, W_MAX)

# -------------------------------------------------
# 4. TVTP without hysteresis (probability-weighted Kelly * eta_t, then cap)
#    We rebuild a simple version here for fairness
# -------------------------------------------------
prob_safe = np.clip(regime_prob_smoothed.values, 1e-12, 1.0)
entropy = -np.sum(prob_safe * np.log(prob_safe), axis=1)
eta_t_simple = 0.5 * np.exp(-1.5 * entropy)            # original conservative settings

w_tvtp = pd.Series(
    (regime_prob_smoothed.values @ kelly_raw) * eta_t_simple,
    index=price_data.index
).clip(W_MIN, W_MAX)

# -------------------------------------------------
# 5. Master Strategy positions (already exist from earlier cells)
# -------------------------------------------------
# strategy_position should still be in memory from Cell 9/10
strategy_position = position_hysteresis.clip(W_MIN, W_MAX)

# -------------------------------------------------
# 6. Out-of-Sample returns
# -------------------------------------------------
bh_oos = price_data["SPY_return"].iloc[TEST_START:]

def oos_ret(weight):
    return (weight.iloc[TEST_START:].shift(1) * bh_oos).fillna(0.0)

oos_fixed_pos = strategy_position.iloc[TEST_START:]
oos_fixed_ret = oos_ret(strategy_position)

# Rolling re-estimation (if available)
rolling_available = False
if os.path.exists("rolling_mle_checkpoint.pkl"):
    with open("rolling_mle_checkpoint.pkl", "rb") as f:
        ckpt = pickle.load(f)
    oos_roll_pos = ckpt["oos_positions"].iloc[TEST_START:].fillna(strategy_position.iloc[TEST_START:])
    oos_roll_pos = oos_roll_pos.clip(W_MIN, W_MAX)
    oos_roll_ret = (oos_roll_pos.shift(1) * bh_oos).fillna(0.0)
    rolling_available = True
    print("Rolling re-estimation checkpoint loaded and capped.")
else:
    print("No rolling checkpoint found – skipping rolling column.")

# -------------------------------------------------
# 7. Performance table
# -------------------------------------------------
oos_list = [
    performance_summary(bh_oos,               name="1. Buy-and-Hold (OOS)"),
    performance_summary(oos_ret(w_const_vol), name="2. Constant Vol Targeting (OOS)"),
    performance_summary(oos_ret(w_cond_vol),  name="3. Conditional Vol Targeting (OOS)"),
    performance_summary(oos_ret(w_std_ms),    name="4. Standard MS-GARCH (filtered, capped)"),
    performance_summary(oos_ret(w_tvtp),      name="5. TVTP (no hysteresis, capped)"),
    performance_summary(oos_fixed_ret,        name="6. Master Strategy – fixed rule (OOS)"),
]

if rolling_available:
    oos_list.append(
        performance_summary(oos_roll_ret, name="7. Master Strategy – rolling re-est. (OOS)")
    )

# Filter out any None results and set correct index
oos_df = pd.DataFrame([x for x in oos_list if x is not None]).set_index("Name")

print("=" * 110)
print(f"CLEAN Out-of-Sample Performance (after {TEST_START} days, W_MAX={W_MAX})")
print("=" * 110)
print(oos_df.round(4).to_string())
print()

oos_df.to_csv("oos_performance_clean.csv")
print("Saved → oos_performance_clean.csv")

# -------------------------------------------------
# 8. Quick exposure diagnostics
# -------------------------------------------------
print("\n----- Average Exposure (OOS) -----")
print(f"Constant Vol     : {w_const_vol.iloc[TEST_START:].mean():.3f}")
print(f"Conditional Vol  : {w_cond_vol.iloc[TEST_START:].mean():.3f}")
print(f"Standard MS      : {w_std_ms.iloc[TEST_START:].mean():.3f}")
print(f"TVTP no hyst     : {w_tvtp.iloc[TEST_START:].mean():.3f}")
print(f"Master fixed     : {oos_fixed_pos.mean():.3f}")
if rolling_available:
    print(f"Master rolling   : {oos_roll_pos.mean():.3f}")


## 16. Additional Analysis

In [ ]:
# Out-of-Sample Equity Curves, Positions & Drawdowns

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# -------------------------------------------------
# Prepare series (reuse variables from previous cell)
# -------------------------------------------------
# Equity curves
eq_bh      = (1 + bh_oos).cumprod()
eq_fixed   = (1 + oos_fixed_ret).cumprod()
eq_tvtp    = (1 + oos_ret(w_tvtp)).cumprod()

if rolling_available:
    eq_roll = (1 + oos_roll_ret).cumprod()
else:
    eq_roll = None

# Drawdowns
def compute_drawdown(equity):
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return dd

dd_bh    = compute_drawdown(eq_bh)
dd_fixed = compute_drawdown(eq_fixed)
dd_tvtp  = compute_drawdown(eq_tvtp)
if rolling_available:
    dd_roll = compute_drawdown(eq_roll)

# -------------------------------------------------
# Plot
# -------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)

# ----- Panel 1: Cumulative Wealth (log scale) -----
axes[0].plot(eq_bh.index, eq_bh, label="Buy-and-Hold", color="navy", lw=1.8)
axes[0].plot(eq_fixed.index, eq_fixed, label="Master (fixed rule)", color="darkgreen", lw=1.8)
if rolling_available:
    axes[0].plot(eq_roll.index, eq_roll, label="Master (rolling re-est.)", color="darkorange", lw=1.8, ls="--")
axes[0].plot(eq_tvtp.index, eq_tvtp, label="TVTP (no hysteresis)", color="purple", lw=1.3, alpha=0.8)

axes[0].set_ylabel("Cumulative Wealth")
axes[0].set_title("Out-of-Sample Equity Curves")
axes[0].legend(loc="upper left")
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

# ----- Panel 2: Position Path -----
axes[1].plot(oos_fixed_pos.index, oos_fixed_pos, label="Master (fixed)", color="darkgreen", lw=1.2)
if rolling_available:
    axes[1].plot(oos_roll_pos.index, oos_roll_pos, label="Master (rolling)", color="darkorange", lw=1.1, alpha=0.85)
axes[1].axhline(1.0, color="gray", ls=":", lw=1)
axes[1].axhline(0.0, color="gray", ls=":", lw=1)
axes[1].set_ylabel("Position Weight")
axes[1].set_title("Strategy Position Over Time")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

# ----- Panel 3: Drawdown -----
axes[2].plot(dd_bh.index, dd_bh, label="Buy-and-Hold", color="navy", lw=1.5)
axes[2].plot(dd_fixed.index, dd_fixed, label="Master (fixed)", color="darkgreen", lw=1.5)
if rolling_available:
    axes[2].plot(dd_roll.index, dd_roll, label="Master (rolling)", color="darkorange", lw=1.5, ls="--")
axes[2].plot(dd_tvtp.index, dd_tvtp, label="TVTP (no hysteresis)", color="purple", lw=1.2, alpha=0.8)

axes[2].set_ylabel("Drawdown")
axes[2].set_xlabel("Date")
axes[2].set_title("Out-of-Sample Drawdowns")
axes[2].legend(loc="lower left")
axes[2].grid(True, alpha=0.3)

# Format x-axis
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("oos_equity_position_drawdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → oos_equity_position_drawdown.png")


## 17. Additional Analysis

In [ ]:
# Ledoit & Wolf (2008) Sharpe Ratio Difference Test

import numpy as np
import pandas as pd
from scipy.stats import norm

def ledroit_wolf_sharpe_test(ret1, ret2, hac_lags=None):
    """
    One-sided test of H0: Sharpe1 <= Sharpe2  vs  H1: Sharpe1 > Sharpe2
    Returns: delta_sharpe, se, t_stat, p_value
    """
    r1 = np.asarray(ret1).flatten()
    r2 = np.asarray(ret2).flatten()
    n = len(r1)
    
    mu1, mu2 = r1.mean(), r2.mean()
    sig1, sig2 = r1.std(ddof=1), r2.std(ddof=1)
    
    sr1 = mu1 / sig1 * np.sqrt(252)
    sr2 = mu2 / sig2 * np.sqrt(252)
    delta = sr1 - sr2
    
    # Moment vector for delta method
    # theta = [mu1, mu2, sig1^2, sig2^2, cov12]
    cov12 = np.cov(r1, r2, ddof=1)[0, 1]
    
    # Gradient of annualized Sharpe difference
    # Simplified implementation following LW (2008) Appendix
    g = np.array([
        1/sig1,
        -1/sig2,
        -0.5 * mu1 / (sig1**3),
        0.5 * mu2 / (sig2**3),
        0.0
    ]) * np.sqrt(252)
    
    # Sample moments
    m1 = r1 - mu1
    m2 = r2 - mu2
    v1 = r1**2 - sig1**2
    v2 = r2**2 - sig2**2
    c12 = r1*r2 - cov12
    
    moments = np.column_stack([m1, m2, v1, v2, c12])
    
    # HAC variance (Newey-West)
    if hac_lags is None:
        hac_lags = int(np.floor(4 * (n/100)**(2/9)))
    
    gamma0 = moments.T @ moments / n
    var = gamma0.copy()
    for lag in range(1, hac_lags + 1):
        w = 1.0 - lag / (hac_lags + 1)
        gamma = moments[lag:].T @ moments[:-lag] / n
        var += w * (gamma + gamma.T)
    
    se = np.sqrt(g.T @ var @ g / n)
    t_stat = delta / se if se > 0 else np.nan
    p_value = 1 - norm.cdf(t_stat)   # one-sided
    
    return {
        "Sharpe_1": sr1,
        "Sharpe_2": sr2,
        "Delta": delta,
        "SE": se,
        "t_stat": t_stat,
        "p_value": p_value
    }

# -------------------------------------------------
# Run tests: Master strategies vs Buy-and-Hold
# -------------------------------------------------
print("=" * 90)
print("Ledoit-Wolf Sharpe Ratio Difference Tests (OOS)")
print("H0: Sharpe(Strategy) <= Sharpe(Buy-and-Hold)")
print("=" * 90)

tests = []

# Master fixed vs B&H
res_fixed = ledroit_wolf_sharpe_test(oos_fixed_ret, bh_oos)
tests.append({"Comparison": "Master fixed vs B&H", **res_fixed})

# Master rolling vs B&H
if rolling_available:
    res_roll = ledroit_wolf_sharpe_test(oos_roll_ret, bh_oos)
    tests.append({"Comparison": "Master rolling vs B&H", **res_roll})

# TVTP vs B&H
res_tvtp = ledroit_wolf_sharpe_test(oos_ret(w_tvtp), bh_oos)
tests.append({"Comparison": "TVTP (no hyst) vs B&H", **res_tvtp})

# Master rolling vs Master fixed
if rolling_available:
    res_roll_vs_fixed = ledroit_wolf_sharpe_test(oos_roll_ret, oos_fixed_ret)
    tests.append({"Comparison": "Master rolling vs Master fixed", **res_roll_vs_fixed})

test_df = pd.DataFrame(tests).set_index("Comparison")
print(test_df.round(4).to_string())
print()
test_df.to_csv("ledoit_wolf_tests.csv")
print("Saved → ledoit_wolf_tests.csv")


## 18. Additional Analysis

In [ ]:
# Monte Carlo Simulation – Full Re-estimation (Finite Sample Properties)
# Full extension: each simulated path is re-estimated with the same
# TVTP-MS-GJR-GARCH MLE procedure (multiple random starts + dual identification).
# Metrics: parameter Bias / RMSE, regime classification Macro F1-Score.
#
# If a previously saved result exists, it is loaded instead of re-running.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from scipy.optimize import minimize
from scipy.special import logsumexp
from tqdm.notebook import tqdm
import os
import warnings
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
N_MC          = 40
T_LIST        = [500, 1000, 2500]
N_STARTS_MC   = 6
MAX_ITER_MC   = 400
RANDOM_SEED   = 42
K             = 3
FORCE_REEST   = False          # set True only when you want to force re-run
SAVE_PATH     = "monte_carlo_full_results.npz"

# True DGP = real-data MLE estimates (already in memory)
true_mu              = mu.copy()
true_omega           = omega.copy()
true_alpha           = alpha.copy()
true_gamma           = gamma.copy()
true_beta            = beta.copy()
true_trans_intercept = trans_intercept.copy()
true_trans_slope     = trans_slope.copy()

print("True DGP parameters (from real-data MLE):")
print(f"  mu     = {np.round(true_mu, 6)}")
print(f"  omega  = {np.round(true_omega, 8)}")
print(f"  alpha  = {np.round(true_alpha, 4)}")
print(f"  beta   = {np.round(true_beta, 4)}")
print()

# ---------------------------------------------------------------------------
# Load previous result if available
# ---------------------------------------------------------------------------
if (not FORCE_REEST) and os.path.exists(SAVE_PATH):
    data = np.load(SAVE_PATH, allow_pickle=True)
    results = data["results"].item()
    mc_df   = pd.DataFrame(data["summary_dict"].item()).set_index("T")

    print("=" * 80)
    print(f"Loaded previous Monte Carlo results from {SAVE_PATH}")
    print(f"N_MC              : {data['N_MC']}")
    print(f"N_STARTS_MC       : {data['N_STARTS_MC']}")
    print(f"T_LIST            : {list(data['T_LIST'])}")
    print(f"Timestamp         : {data['timestamp']}")
    print("=" * 80)
    print(mc_df.round(5).to_string())
    print()

else:
    # -----------------------------------------------------------------------
    # 1. Simulate paths from the true TVTP-MS-GJR-GARCH DGP
    # -----------------------------------------------------------------------
    def simulate_tvtp_ms_gjr(T, mu, omega, alpha, gamma, beta,
                            trans_intercept, trans_slope, seed=None):
        if seed is not None:
            np.random.seed(seed)

        inv = np.zeros(T)
        inv[0] = 1.0
        for t in range(1, T):
            inv[t] = 0.92 * inv[t-1] + 0.08 + 0.035 * np.random.randn()
            inv[t] = np.clip(inv[t], 0.65, 1.80)

        returns = np.zeros(T)
        states  = np.zeros(T, dtype=int)
        sigma2  = np.full(K, 1e-4)

        states[0]  = 0
        returns[0] = mu[0] + np.sqrt(sigma2[0]) * np.random.randn()

        for t in range(1, T):
            P = transition_matrix(inv[t-1], trans_intercept, trans_slope)
            states[t] = np.random.choice(K, p=P[states[t-1]])

            eps_prev = returns[t-1] - mu[states[t-1]]
            I_neg = 1.0 if eps_prev < 0 else 0.0
            k = states[t]
            sigma2[k] = (omega[k]
                         + (alpha[k] + gamma[k] * I_neg) * eps_prev**2
                         + beta[k] * sigma2[k])
            sigma2[k] = max(sigma2[k], 1e-12)
            returns[t] = mu[k] + np.sqrt(sigma2[k]) * np.random.randn()

        return returns, states, inv

    # -----------------------------------------------------------------------
    # 2. Negative log-likelihood for a given simulated sample
    # -----------------------------------------------------------------------
    def neg_loglik_sim(theta, ret, inv_lag):
        mu_u, omega_u, alpha_u, gamma_u, beta_u, t_int, t_slope = unpack_parameters(theta)

        if not (omega_u[0] < omega_u[1] < omega_u[2]):
            return 1e12 + 1e6 * np.sum(np.maximum(0.0, omega_u[:-1] - omega_u[1:]))

        T_sim = len(ret)
        xi_prev     = np.ones(K) / K
        sigma2_prev = np.full(K, np.var(ret[:min(50, T_sim)]))
        log_lik     = 0.0

        for t in range(T_sim):
            P = transition_matrix(inv_lag[t], t_int, t_slope)
            xi_pred = P.T @ xi_prev

            dens = np.zeros(K)
            for k in range(K):
                dens[k] = (1.0 / np.sqrt(2 * np.pi * sigma2_prev[k])
                           * np.exp(-0.5 * (ret[t] - mu_u[k])**2 / sigma2_prev[k]))
            dens = np.maximum(dens, 1e-300)
            lik  = np.dot(xi_pred, dens)
            log_lik += np.log(lik + 1e-300)

            xi_prev = (xi_pred * dens) / (lik + 1e-300)

            for k in range(K):
                eps = ret[t] - mu_u[k]
                I_neg = 1.0 if eps < 0 else 0.0
                sigma2_prev[k] = (omega_u[k]
                                  + (alpha_u[k] + gamma_u[k] * I_neg) * eps**2
                                  + beta_u[k] * sigma2_prev[k])
                sigma2_prev[k] = max(sigma2_prev[k], 1e-12)

        return -log_lik

    # -----------------------------------------------------------------------
    # 3. Fast multi-start MLE for one simulated path
    # -----------------------------------------------------------------------
    def estimate_one_path(ret, inv_lag, n_starts=N_STARTS_MC):
        best_nll = np.inf
        best_theta = None

        for s in range(n_starts):
            np.random.seed(RANDOM_SEED + 1000 + s)

            mu_init    = true_mu + np.random.normal(0, 0.0004, K)
            omega_init = np.log(true_omega * (0.5 + np.random.rand(K)))
            alpha_init = np.log(np.clip(true_alpha * (0.5 + np.random.rand(K)), 1e-6, 0.5))
            gamma_init = true_gamma + np.random.normal(0, 0.04, K)
            beta_raw   = np.clip(true_beta + np.random.normal(0, 0.05, K), 0.50, 0.97)
            beta_init  = np.log(beta_raw / (1 - beta_raw))
            trans_init = np.random.normal(0, 0.4, K * K * 2)

            theta0 = np.concatenate([mu_init, omega_init, alpha_init,
                                     gamma_init, beta_init, trans_init])

            try:
                res = minimize(neg_loglik_sim, theta0,
                               args=(ret, inv_lag),
                               method="L-BFGS-B",
                               options={"maxiter": MAX_ITER_MC, "ftol": 1e-8})
                if res.success and res.fun < best_nll:
                    best_nll = res.fun
                    best_theta = res.x.copy()
            except Exception:
                continue

        if best_theta is None:
            return None

        mu_e, omega_e, alpha_e, gamma_e, beta_e, t_int_e, t_slope_e = unpack_parameters(best_theta)
        return {
            "mu": mu_e,
            "omega": omega_e,
            "alpha": alpha_e,
            "gamma": gamma_e,
            "beta": beta_e,
            "trans_intercept": t_int_e,
            "trans_slope": t_slope_e,
            "nll": best_nll
        }

    # -----------------------------------------------------------------------
    # 4. Filtered most-likely state sequence (for F1)
    # -----------------------------------------------------------------------
    def filtered_states(ret, inv_lag, est):
        mu_e, omega_e, alpha_e, gamma_e, beta_e = (est["mu"], est["omega"], est["alpha"],
                                                    est["gamma"], est["beta"])
        t_int, t_slope = est["trans_intercept"], est["trans_slope"]

        T_sim = len(ret)
        xi_prev     = np.ones(K) / K
        sigma2_prev = np.full(K, np.var(ret[:min(50, T_sim)]))
        states_hat  = np.zeros(T_sim, dtype=int)

        for t in range(T_sim):
            P = transition_matrix(inv_lag[t], t_int, t_slope)
            xi_pred = P.T @ xi_prev

            dens = np.zeros(K)
            for k in range(K):
                dens[k] = (1.0 / np.sqrt(2 * np.pi * sigma2_prev[k])
                           * np.exp(-0.5 * (ret[t] - mu_e[k])**2 / sigma2_prev[k]))
            dens = np.maximum(dens, 1e-300)
            lik  = np.dot(xi_pred, dens)
            xi_prev = (xi_pred * dens) / (lik + 1e-300)
            states_hat[t] = np.argmax(xi_prev)

            for k in range(K):
                eps = ret[t] - mu_e[k]
                I_neg = 1.0 if eps < 0 else 0.0
                sigma2_prev[k] = (omega_e[k]
                                  + (alpha_e[k] + gamma_e[k] * I_neg) * eps**2
                                  + beta_e[k] * sigma2_prev[k])
                sigma2_prev[k] = max(sigma2_prev[k], 1e-12)

        return states_hat

    # -----------------------------------------------------------------------
    # 5. Monte Carlo loop with progress bar
    # -----------------------------------------------------------------------
    results = {T: {"f1": [], "mu_bias": [], "mu_rmse": [],
                   "omega_bias": [], "omega_rmse": []} for T in T_LIST}

    total_jobs = len(T_LIST) * N_MC
    print(f"Running FULL re-estimation Monte Carlo")
    print(f"  N_MC = {N_MC} | starts per path = {N_STARTS_MC} | T_LIST = {T_LIST}")
    print(f"  Total replications = {total_jobs}")
    print("Progress bar below tracks every completed replication.\n")

    pbar = tqdm(total=total_jobs, desc="Monte Carlo", unit="rep")

    for T in T_LIST:
        f1_list, mu_bias_list, mu_rmse_list = [], [], []
        omega_bias_list, omega_rmse_list = [], []

        for mc in range(N_MC):
            sim_ret, true_states, sim_inv = simulate_tvtp_ms_gjr(
                T, true_mu, true_omega, true_alpha, true_gamma, true_beta,
                true_trans_intercept, true_trans_slope,
                seed=RANDOM_SEED + mc * 17
            )
            inv_lag = np.roll(sim_inv, 1)
            inv_lag[0] = sim_inv[0]

            est = estimate_one_path(sim_ret, inv_lag, n_starts=N_STARTS_MC)
            if est is None:
                pbar.update(1)
                continue

            pred_states = filtered_states(sim_ret, inv_lag, est)
            f1 = f1_score(true_states, pred_states, average="macro", zero_division=0)
            f1_list.append(f1)

            mu_bias_list.append(est["mu"] - true_mu)
            mu_rmse_list.append((est["mu"] - true_mu)**2)
            omega_bias_list.append(est["omega"] - true_omega)
            omega_rmse_list.append((est["omega"] - true_omega)**2)

            pbar.set_postfix({"T": T, "F1": f"{f1:.3f}"})
            pbar.update(1)

        results[T]["f1"] = f1_list
        results[T]["mu_bias"] = mu_bias_list
        results[T]["mu_rmse"] = mu_rmse_list
        results[T]["omega_bias"] = omega_bias_list
        results[T]["omega_rmse"] = omega_rmse_list

        print(f"T={T:4d} | Mean Macro F1 = {np.mean(f1_list):.3f} | "
              f"Mu RMSE = {np.sqrt(np.mean(mu_rmse_list)):.6f}")

    pbar.close()
    print("\nMonte Carlo finished.")

    # -----------------------------------------------------------------------
    # 6. Summary table
    # -----------------------------------------------------------------------
    summary = []
    for T in T_LIST:
        mu_bias = np.mean(results[T]["mu_bias"], axis=0) if results[T]["mu_bias"] else np.zeros(K)
        mu_rmse = np.sqrt(np.mean(results[T]["mu_rmse"], axis=0)) if results[T]["mu_rmse"] else np.zeros(K)
        summary.append({
            "T": T,
            "Mean_Macro_F1": np.mean(results[T]["f1"]) if results[T]["f1"] else np.nan,
            "Std_Macro_F1": np.std(results[T]["f1"]) if results[T]["f1"] else np.nan,
            "Mu0_Bias": mu_bias[0],
            "Mu1_Bias": mu_bias[1],
            "Mu2_Bias": mu_bias[2],
            "Mu0_RMSE": mu_rmse[0],
            "Mu1_RMSE": mu_rmse[1],
            "Mu2_RMSE": mu_rmse[2],
        })

    mc_df = pd.DataFrame(summary).set_index("T")
    print("\n" + "=" * 80)
    print("FULL Re-estimation Monte Carlo – Finite Sample Properties")
    print("=" * 80)
    print(mc_df.round(5).to_string())
    mc_df.to_csv("monte_carlo_full_summary.csv")
    print("\nSaved → monte_carlo_full_summary.csv")

    # Save full results for future sessions
    np.savez(
        SAVE_PATH,
        results      = results,
        summary_dict = {c: mc_df.reset_index()[c].values for c in mc_df.reset_index().columns},
        N_MC         = N_MC,
        N_STARTS_MC  = N_STARTS_MC,
        T_LIST       = np.array(T_LIST),
        timestamp    = np.datetime64("now")
    )
    print(f"Full Monte Carlo results saved → {SAVE_PATH}")

# ---------------------------------------------------------------------------
# 7. Plots (always run, whether loaded or freshly computed)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

means = [np.mean(results[T]["f1"]) if results[T]["f1"] else np.nan for T in T_LIST]
stds  = [np.std(results[T]["f1"]) if results[T]["f1"] else np.nan for T in T_LIST]
axes[0].errorbar(T_LIST, means, yerr=stds, fmt="o-", color="darkblue",
                 capsize=5, lw=2, markersize=8)
axes[0].set_xlabel("Sample Size T")
axes[0].set_ylabel("Macro F1-Score")
axes[0].set_title("Regime Classification Accuracy (Full Re-estimation)")
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

f1_data = [results[T]["f1"] for T in T_LIST if results[T]["f1"]]
if f1_data:
    bp = axes[1].boxplot(f1_data, labels=[str(T) for T in T_LIST],
                         patch_artist=True, widths=0.5)
    for patch in bp["boxes"]:
        patch.set_facecolor("lightblue")
axes[1].set_xlabel("Sample Size T")
axes[1].set_ylabel("Macro F1-Score")
axes[1].set_title("Distribution of Macro F1 across MC Replications")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("monte_carlo_full_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → monte_carlo_full_f1.png")


## 19. Additional Analysis